In [1]:
# =============================================================================
# CELL 1: CONFIGURATION
# =============================================================================

granularity = 'w'  # 'q' = quarterly, 'm' = monthly, 'w' = weekly

START_DATE = '2026-01-01'
END_DATE = None

run_every_query = False  # True = run SQL; False = use cache/ pickles from bareboned_ragu_new

DATE_COL_MAP = {'q': 'book_date', 'm': 'book_date', 'w': 'app_date'}

LOBS = ['AN', 'FRN', 'STG', 'FLD', 'ENT']

BASELINES = {
    'AN':  {'ltv': 1.94, 'new_recovery_unadjusted': 0.58, 'apr': 0.25},
    'FRN': {'ltv': 1.94, 'new_recovery_unadjusted': 0.54, 'apr': 0.25},
    'STG': {'ltv': 1.94, 'new_recovery_unadjusted': 0.60, 'apr': 0.25},
    'FLD': {'ltv': 1.45, 'new_recovery_unadjusted': 0.54, 'apr': 0.235},
    'ENT': {'ltv': 1.45, 'new_recovery_unadjusted': 0.55, 'apr': 0.235},
}

MODEL_PARAMS = {
    'mean_unit_loss': 0.5,
    'unit_loss_to_model_score': 0.02,
}

DIAG_N_VINTAGES = 6

EXCEL_OUTPUT = 'nonkmx_gl_diagnostics.xlsx'

In [2]:
# Parameters
granularity = "w"
START_DATE = "2026-01-01"
END_DATE = None
run_every_query = False
LOBS = ["AN", "FRN", "STG", "FLD", "ENT"]
BASELINES = {"AN": {"ltv": 1.94, "new_recovery_unadjusted": 0.58, "apr": 0.25}, "FRN": {"ltv": 1.94, "new_recovery_unadjusted": 0.54, "apr": 0.25}, "STG": {"ltv": 1.94, "new_recovery_unadjusted": 0.6, "apr": 0.25}, "FLD": {"ltv": 1.45, "new_recovery_unadjusted": 0.54, "apr": 0.235}, "ENT": {"ltv": 1.45, "new_recovery_unadjusted": 0.55, "apr": 0.235}}
MODEL_PARAMS = {"mean_unit_loss": 0.5, "unit_loss_to_model_score": 0.02}
DIAG_N_VINTAGES = 6


In [3]:
# =============================================================================
# CELL 2: IMPORTS AND DERIVED CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import math
import os
import openpyxl

pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)

date_col = DATE_COL_MAP[granularity]
start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp.today().normalize()

PERIOD_FREQ_MAP = {'q': 'Q', 'm': 'M', 'w': 'W-SAT'}
period_freq = PERIOD_FREQ_MAP[granularity]

start_period = start_date.to_period(period_freq)
end_period = end_date.to_period(period_freq)

min_date_sql = f"'{START_DATE}'"

print(f"Granularity: {granularity}")
print(f"Date column: {date_col}")
print(f"Period range: {start_period} to {end_period}")

Granularity: w
Date column: app_date
Period range: 2025-12-28/2026-01-03 to 2026-05-03/2026-05-09


In [4]:
# =============================================================================
# CELL 3: UTILITY FUNCTIONS
# =============================================================================

def run_sql(filename, sub_list=None, connection=None, filename_is_query=False):
    if sub_list is None:
        sub_list = []
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for text, var in sub_list:
        query = query.replace(text, var)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df


def store_pickle(data, filename):
    if isinstance(data, str):
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


def cached_sql(filename, pickle_name, sub_list=None, connection=None, force_refresh=False):
    if force_refresh or not os.path.exists(pickle_name):
        df = run_sql(filename, sub_list=sub_list, connection=connection)
        store_pickle(df, pickle_name)
        return df
    return get_pickle(pickle_name)


def weighted_average_and_sum(group, metrics):
    if isinstance(metrics, str):
        weighted_avg = (group[metrics] * group.amt_financed).sum() / group.amt_financed.sum()
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        weighted_avg = (group[metric] * group.amt_financed).sum() / group.amt_financed.sum()
        result_dict[metric] = weighted_avg
    return pd.Series(result_dict)


def format_vintage(period_series):
    if len(period_series) == 0:
        return period_series.astype(str)
    freq = period_series.iloc[0].freqstr
    if freq == 'Q-DEC':
        return period_series.dt.year.astype(str) + ' Q' + period_series.dt.quarter.astype(str)
    elif freq == 'M':
        return period_series.dt.year.astype(str) + ' M' + period_series.dt.month.astype(str).str.zfill(2)
    return period_series.astype(str)

In [5]:
# =============================================================================
# CELL 4: ULA MULTIPLIER FUNCTIONS (NONKMX)
# =============================================================================

def get_ula_multiplier_nonkmx(ula_df, leave_out='None'):
    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag
    if leave_out != 'Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)
    if leave_out != 'Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)
    if leave_out != 'High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag
    if leave_out != 'High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)
    if leave_out != 'Car make':
        ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                                   - 0.1 * ula_df.car_make_benefit_flag
                                   - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)
    if leave_out != 'Theft risk':
        ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                                   + 0.013 * ula_df.pricing_change_flag)
    if leave_out != 'MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag
    if leave_out != 'Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag
    if leave_out != 'Student Loans':
        ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
            - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
            * (1 - ula_df.student_loan_flag)) * ula_df.student_loans_cutoff_date
    if leave_out != 'Low PTI':
        ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag
    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag
    if leave_out != 'Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag
    if leave_out != 'Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag
    if leave_out != 'Fraud':
        ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)
    if leave_out != 'Driver flag':
        ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag
    if leave_out != 'Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)
    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier *= ula_df.pricing_scalar
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)

    return ula_df


def get_ula_multiplier_nonkmx_diag(ula_df, leave_out='None', verbose=True):
    """Identical logic to get_ula_multiplier_nonkmx, plus per-step mean tracking."""
    steps = {}
    def _record(label, df):
        steps[label] = df.loss_multiplier.mean()

    ula_df['loss_multiplier'] = 1.0
    _record('00_initial', ula_df)
    if verbose:
        print(f"00_initial  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag
    _record('01_prev_aca_chargeoff', ula_df)
    if verbose:
        print(f"01_prev_co  | flag mean: {ula_df.prev_co_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)
    _record('02_small_amt_financed', ula_df)
    if verbose:
        print(f"02_small_af | flag mean: {ula_df.small_amt_financed_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)
    _record('03_zero_cash_down', ula_df)
    if verbose:
        print(f"03_zero_cd  | flag mean: {ula_df.zero_cash_down_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag
    _record('04_high_mileage', ula_df)
    if verbose:
        print(f"04_high_mi  | flag mean: {ula_df.high_mileage_vehicle_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)
    _record('05_high_pti', ula_df)
    if verbose:
        print(f"05_high_pti | flag mean: {ula_df.high_pti_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Car make':
        ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                                   - 0.1 * ula_df.car_make_benefit_flag
                                   - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)
    _record('06_car_make', ula_df)
    if verbose:
        print(f"06_car_make | penalty: {ula_df.car_make_penalty_flag.mean():.6f}  benefit: {ula_df.car_make_benefit_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Theft risk':
        ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                                   + 0.013 * ula_df.pricing_change_flag)
    _record('07_theft_risk', ula_df)
    if verbose:
        print(f"07_theft    | flag mean: {ula_df.theft_risk_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag
    _record('08_mcy_low_mileage', ula_df)
    if verbose:
        print(f"08_mcy_low  | flag mean: {ula_df.mcy_low_mileage_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag
    _record('09_weekend_weekday', ula_df)
    if verbose:
        print(f"09_wknd/day | weekend: {ula_df.weekend_flag.mean():.6f}  weekday: {ula_df.weekday_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Student Loans':
        ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
            - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
            * (1 - ula_df.student_loan_flag)) * (ula_df.student_loans_cutoff_date)
    _record('10_student_loans', ula_df)
    if verbose:
        print(f"10_stud_ln  | flag mean: {ula_df.student_loan_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Low PTI':
        ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag
    _record('11_low_pti', ula_df)
    if verbose:
        print(f"11_low_pti  | flag mean: {ula_df.low_pti_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag
    _record('12_chime', ula_df)
    if verbose:
        print(f"12_chime    | flag mean: {ula_df.nonkmx_chime_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag
    _record('13_employment_type', ula_df)
    if verbose:
        n = len(ula_df)
        print(f"13_employ   | seasonal: {ula_df.seasonal_employment_flag.sum()/n:.6f}  waiter: {ula_df.waiter_employment_flag.sum()/n:.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag
    _record('14_auth_tradelines', ula_df)
    if verbose:
        print(f"14_auth_tl  | flag mean: {ula_df.nonkmx_auth_tradelines_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Fraud':
        ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)
    _record('15_fraud', ula_df)
    if verbose:
        print(f"15_fraud    | fraud_adj mean: {ula_df.fraud_adjustment.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Driver flag':
        ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag
    _record('16_driver_flag', ula_df)
    if verbose:
        print(f"16_driver   | flag mean: {ula_df.driver_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)

    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier *= ula_df.pricing_scalar
    _record('17_pricing_scalar', ula_df)
    if verbose:
        print(f"17_pricing  | scalar mean: {ula_df.pricing_scalar.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)
    _record('18_final', ula_df)
    if verbose:
        print(f"18_final    | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}  (post all clips)")

    n = len(ula_df)
    flags = {
        'prev_co_flag': ula_df.prev_co_flag.mean(),
        'small_amt_financed_flag': ula_df.small_amt_financed_flag.mean(),
        'pricing_change_flag': ula_df.pricing_change_flag.mean(),
        'zero_cash_down_flag': ula_df.zero_cash_down_flag.mean(),
        'high_mileage_vehicle_flag': ula_df.high_mileage_vehicle_flag.mean(),
        'high_pti_flag': ula_df.high_pti_flag.mean(),
        'car_make_penalty_flag': ula_df.car_make_penalty_flag.mean(),
        'car_make_benefit_flag': ula_df.car_make_benefit_flag.mean(),
        'theft_risk_flag': ula_df.theft_risk_flag.mean(),
        'mcy_low_mileage_flag': ula_df.mcy_low_mileage_flag.mean(),
        'weekend_flag': ula_df.weekend_flag.mean(),
        'weekday_flag': ula_df.weekday_flag.mean(),
        'student_loan_flag': ula_df.student_loan_flag.mean(),
        'student_loans_cutoff_date': ula_df.student_loans_cutoff_date.mean(),
        'low_pti_flag': ula_df.low_pti_flag.mean(),
        'nonkmx_chime_flag': ula_df.nonkmx_chime_flag.mean(),
        'seasonal_employment_pct': ula_df.seasonal_employment_flag.sum() / n,
        'waiter_employment_pct': ula_df.waiter_employment_flag.sum() / n,
        'nonkmx_auth_tradelines_flag': ula_df.nonkmx_auth_tradelines_flag.mean(),
        'fraud_adjustment_mean': ula_df.fraud_adjustment.mean(),
        'driver_flag': ula_df.driver_flag.mean(),
        'pricing_scalar_mean': ula_df.pricing_scalar.mean(),
    }

    return ula_df, pd.Series(steps), pd.Series(flags)

In [6]:
# =============================================================================
# CELL 5: DATA FETCH (shared cache/ from bareboned_ragu_new)
# =============================================================================

os.makedirs('cache', exist_ok=True)
force = run_every_query

need_conn = force or not all(
    os.path.exists(p) for p in ('cache/ula_v1.pkl', 'cache/dla_v1.pkl', 'cache/new_recovery_v1.pkl')
)

if need_conn:
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        ula_df_total = cached_sql(
            'vintage_level_ula_query.txt', 'cache/ula_v1.pkl',
            sub_list=[('{min_book_date}', min_date_sql)], connection=conn, force_refresh=force,
        )
        print('ULA ready')
        dla_df = cached_sql(
            'new_dll_query.txt', 'cache/dla_v1.pkl',
            connection=conn, force_refresh=force,
        )
        print('DLA ready')
        new_recovery = cached_sql(
            'new_recovery_queryt.txt', 'cache/new_recovery_v1.pkl',
            connection=conn, force_refresh=force,
        )
        print('New recovery ready')
else:
    ula_df_total = get_pickle('cache/ula_v1.pkl')
    dla_df = get_pickle('cache/dla_v1.pkl')
    new_recovery = get_pickle('cache/new_recovery_v1.pkl')
    print('ULA, DLA, New recovery loaded from cache')

print(f"ULA records: {len(ula_df_total):,}")
print("[PROGRESS] Data Fetch Complete")


ULA, DLA, New recovery loaded from cache
ULA records: 752,571
[PROGRESS] Data Fetch Complete


In [7]:
# =============================================================================
# CELL 6: PERIOD ASSIGNMENT, FILTERING, FLAG CONSTRUCTION
# =============================================================================

ula_df_total = ula_df_total[ula_df_total.lob != 'Core']

ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
ula_df_total['book_date'] = pd.to_datetime(ula_df_total['book_date'])
new_recovery['app_date'] = pd.to_datetime(new_recovery['app_date'])
new_recovery['book_date'] = pd.to_datetime(new_recovery['book_date'])

for df in [ula_df_total, new_recovery]:
    df['quarter'] = pd.to_datetime(df[date_col]).dt.to_period('Q')
    df['month'] = pd.to_datetime(df[date_col]).dt.to_period('M')
    df['week'] = pd.to_datetime(df[date_col]).dt.to_period('W-SAT')
    period_key = {'q': 'quarter', 'm': 'month', 'w': 'week'}
    df['period'] = df[period_key[granularity]]

for df in [ula_df_total, new_recovery]:
    mask = (df['period'] >= start_period) & (df['period'] <= end_period)
    df.drop(df[~mask].index, inplace=True)

for df in [ula_df_total, new_recovery]:
    df['book_week'] = df['book_week'].astype(str)
    df['app_week'] = df['app_week'].astype(str)

date_col_str = f'{date_col}_str'
ula_df_total[date_col_str] = ula_df_total[date_col].astype(str)

# --- ULA Processing ---
ula_df_total['vehicle_age'] = np.maximum(ula_df_total[date_col_str].str[:4].astype(int) - ula_df_total.model_year, 1/365)
ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
ula_df_total.make = ula_df_total.make.str.upper().str[:3]
ula_df_total.lob_or_bucket = np.select(
    [ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
     ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])],
    ['D', 'C'], default=ula_df_total.lob_or_bucket)

# --- DLA Merge ---
dla_df = dla_df.rename(columns={'valid_vintage': 'book_vintage'})
dla_explicit = dla_df[dla_df['book_vintage'] != 'current']
dla_current = dla_df[dla_df['book_vintage'] == 'current'].drop(columns='book_vintage')
last_explicit_vintage = dla_explicit['book_vintage'].max()
ula_df_total = ula_df_total.merge(dla_explicit, how='left', on=['dealer_number', 'book_vintage'])
new_and_missing = ula_df_total['pricing_scalar'].isna() & (ula_df_total['book_vintage'] > last_explicit_vintage)
fallback = ula_df_total.loc[new_and_missing, ['dealer_number']].merge(dla_current, on='dealer_number', how='left')
for col in ['dll_edition', 'loss_ratio', 'dealer_level', 'pricing_scalar']:
    ula_df_total.loc[new_and_missing, col] = fallback[col].values
ula_df_total['pricing_scalar'] = ula_df_total['pricing_scalar'].fillna(1)
ula_df_total.loc[ula_df_total.frni_flag == 1, 'pricing_scalar'] *= 1.05
ula_df_total.loc[ula_df_total.frni_flag == 0, 'pricing_scalar'] *= 0.95

# --- Driver Flag ---
warnings.filterwarnings('ignore', category=UserWarning)
ula_df_total.job_company = ula_df_total.job_company.fillna('not provided')
ula_df_total['driver_flag'] = np.where(
    ula_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
    1, 0)
warnings.filterwarnings('default', category=UserWarning)

# --- NA Handling ---
ula_df_total = ula_df_total.dropna(subset=['sale_price', 'cd_model_score', 'pti', 'lob'])
ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)

# --- Weekly-matching filters ---
ula_df_total['bbltv'] = ula_df_total.amt_financed / ula_df_total.bbvalue.replace(0, np.nan)
ula_df_total = ula_df_total[ula_df_total.amt_financed <= 75000]
ula_df_total = ula_df_total[ula_df_total.pti <= 0.6]
ula_df_total = ula_df_total[
    (ula_df_total.lob == 'MCY') |
    (ula_df_total.bbltv <= 10.0) |
    (ula_df_total.bbvalue.isna()) |
    (ula_df_total.bbvalue == 0)
]
ula_df_total = ula_df_total[ula_df_total.total_income <= 200000]

# --- NonKMX Flags ---
ula_df_total['ent_fld_flag'] = (ula_df_total.lob == 'ENT') | (ula_df_total.lob == 'FLD')
ula_df_total['small_amt_financed_flag'] = (ula_df_total.bbvalue < 5000) & (ula_df_total.amt_financed < 4500) & (ula_df_total.lob != 'MCY')
ula_df_total['zero_cash_down_flag'] = (ula_df_total.cash_down <= 250) & (ula_df_total.tradein_value < 3000)
ula_df_total['high_mileage_vehicle_flag'] = (ula_df_total.mileage >= 100000) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY')
ula_df_total['high_pti_flag'] = (ula_df_total.pti > 0.3) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_penalty_flag'] = ula_df_total.make.isin({'CAD', 'CHR', 'BMW', 'BUI', 'SUB'}) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_benefit_flag'] = (ula_df_total.cd_model_score >= 133) & ula_df_total.make.isin({'HON', 'TOY', 'LEX'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['theft_risk_flag'] = ula_df_total.make.isin({'KIA', 'HYU'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.model_year >= 2015) & (ula_df_total.model_year <= 2021) & (ula_df_total[date_col_str] >= '2022-07-01') & (ula_df_total[date_col_str] < '2025-01-01')
ula_df_total['mcy_low_mileage_flag'] = (ula_df_total.lob == 'MCY') & (ula_df_total.cd_model_score > 140) & (ula_df_total.mileage <= 20000) & (ula_df_total.vehicle_age <= 10)
ula_df_total['weekend_flag'] = ula_df_total.day_of_week.isin([0, 6])
ula_df_total['weekday_flag'] = ula_df_total.day_of_week.isin(range(1, 6))
ula_df_total['nonkmx_chime_flag'] = ula_df_total.nonkmx_chime_indicator
ula_df_total['seasonal_employment_flag'] = ula_df_total['seasonal_employment_flag'] == 1
ula_df_total['waiter_employment_flag'] = ula_df_total['waiter_employment_flag'] == 1
ula_df_total['nonkmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines > 0.2
ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0
ula_df_total['pricing_change_flag'] = ula_df_total[date_col_str] >= '2024-10-01'
ula_df_total['ent_flag'] = (ula_df_total.lob == 'ENT')
ula_df_total['student_loans_cutoff_date'] = ula_df_total[date_col_str] >= '2023-05-01'
ula_df_total['low_pti_flag'] = (ula_df_total.pti <= 0.05) & (ula_df_total.cd_model_score >= 130) & ula_df_total.lob.isin({'AN', 'FLD', 'FRN', 'STG'}) & (ula_df_total.cb_flag)
ula_df_total['student_loan_flag'] = np.where(ula_df_total.student_loan_flag == 1, 1, 0)

# --- Deduplicate driver flags ---
combined_driver_flag_df = ula_df_total.groupby('account_number').driver_flag.max().reset_index()
ula_df_total = ula_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
ula_df_total = ula_df_total.drop(columns='job_company').drop_duplicates()

# --- Filter to nonKMX LOBs only ---
ula_df_total = ula_df_total[ula_df_total.lob.isin(LOBS)]

ula_df_total['vintage'] = format_vintage(ula_df_total['period'])
new_recovery['vintage'] = format_vintage(new_recovery['period'])

ms_df = ula_df_total.groupby(['period', 'lob']).apply(
    weighted_average_and_sum, 'cd_model_score', include_groups=False
).reset_index()
ms_df = ms_df.rename(columns={'cd_model_score': 'model_score'})
ms_df['period'] = format_vintage(ms_df['period'])

print(f"ULA after filters (nonKMX only): {len(ula_df_total):,}")
print(f"LOBs: {sorted(ula_df_total.lob.unique())}")
print(f"Vintages: {ula_df_total.vintage.nunique()}")
print(f"Model scores: {len(ms_df)} period-LOB combinations")

ULA after filters (nonKMX only): 29,763
LOBs: ['AN', 'ENT', 'FLD', 'FRN', 'STG']
Vintages: 19
Model scores: 95 period-LOB combinations


In [8]:
# =============================================================================
# CELL 7: RAGU SCORE COMPUTATION (NONKMX LOBs)
# =============================================================================

mean_unit_loss = MODEL_PARAMS['mean_unit_loss']
unit_loss_to_model_score = MODEL_PARAMS['unit_loss_to_model_score']


def get_ragu_score(vintage, lob, ula_df_total, new_recovery, ms_df, baseline_config, leave_out='None'):
    baseline_ltv = baseline_config['ltv']
    new_baseline_recovery_unadjusted_pct = baseline_config['new_recovery_unadjusted']
    baseline_apr = baseline_config['apr']
    ltv_mult = 17
    apr_mult = 0.7

    ula_df = ula_df_total[(ula_df_total.vintage == vintage) & (ula_df_total.lob == lob)].copy()
    if len(ula_df) == 0:
        return None

    ula_df = get_ula_multiplier_nonkmx(ula_df, leave_out)

    ula_df = ula_df[['account_number', date_col, 'bbvalue', 'sale_price', 'amt_financed', 'lob_or_bucket', 'lob', 'loss_multiplier', 'apr']]

    nr = new_recovery[['account_number', 'new_recovery_multiplier']].drop_duplicates(subset='account_number', keep='first')
    mix_df = ula_df.merge(nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
                          on='account_number', how='left').drop_duplicates(subset='account_number', keep='first')
    mix_df['ltv'] = mix_df.amt_financed / mix_df.bbvalue

    bb_populated_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0)]
    if len(bb_populated_df) == 0:
        return None

    full_pop_metrics = bb_populated_df.groupby('lob').apply(
        weighted_average_and_sum, ['loss_multiplier', 'ltv', 'bbvalue', 'apr'], include_groups=False)

    recovery_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0) & mix_df['recovery_multiplier'].notna()].copy()
    recovery_df['recovery_unadjusted_multiplier'] = recovery_df['recovery_multiplier']
    recovery_metrics = recovery_df.groupby('lob').apply(
        weighted_average_and_sum, ['recovery_unadjusted_multiplier'], include_groups=False)
    recovery_metrics = recovery_metrics.drop(columns='amt_financed')

    grouped_mix_df = full_pop_metrics.join(recovery_metrics)
    vintage_ms_df = ms_df[ms_df['period'] == vintage].copy()

    index_name = grouped_mix_df.index.name
    if isinstance(index_name, str) and index_name in grouped_mix_df.columns:
        grouped_mix_df = grouped_mix_df.reset_index(drop=True)
    else:
        grouped_mix_df = grouped_mix_df.reset_index()

    full_df = grouped_mix_df.merge(vintage_ms_df, on='lob')
    if len(full_df) == 0:
        return None

    full_df['est_unit_loss'] = mean_unit_loss
    full_df['unit_loss_score'] = full_df.model_score + (1 - full_df.loss_multiplier) * full_df.est_unit_loss / unit_loss_to_model_score
    full_df = full_df.set_index('lob')
    full_df['ms_original'] = full_df.model_score.copy()
    full_df['baselined_unadjusted_recovery'] = (full_df.recovery_unadjusted_multiplier / new_baseline_recovery_unadjusted_pct).copy()

    full_df['gross_loss_impact'] = full_df['unit_loss_score'] - full_df['ms_original']
    full_df['recovery_impact'] = (full_df['unit_loss_score'] * full_df['est_unit_loss']
        * full_df['recovery_unadjusted_multiplier'] * (full_df['baselined_unadjusted_recovery'] - 1))
    full_df['ltv_impact'] = ((baseline_ltv / full_df['ltv']) - 1) * ltv_mult
    full_df['apr_impact'] = (baseline_apr - full_df['apr']) / 0.01 * apr_mult
    full_df['ragu_score'] = (
        (1 - full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier) * full_df.unit_loss_score
        + full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier
        * full_df.unit_loss_score * full_df.baselined_unadjusted_recovery
        + full_df['ltv_impact']
        + full_df['apr_impact'])
    full_df['vintage'] = vintage
    return full_df


all_vintages = sorted(ula_df_total['vintage'].unique())
results_by_lob = {}

for lob in LOBS:
    baseline_config = BASELINES[lob]
    lob_results = []
    for vintage in all_vintages:
        try:
            result = get_ragu_score(vintage, lob, ula_df_total, new_recovery, ms_df, baseline_config)
            if result is not None:
                lob_results.append(result)
        except Exception as e:
            print(f"Error: {vintage} {lob}: {e}")
    if lob_results:
        results_by_lob[lob] = pd.concat(lob_results)
        print(f"{lob}: {len(lob_results)} vintages")
    else:
        results_by_lob[lob] = pd.DataFrame()
        print(f"{lob}: no results")

print(f"\nAll LOBs processed.")
print("[PROGRESS] Scoring Complete")


Error: 2026-05-03/2026-05-09 AN: columns overlap but no suffix specified: Index(['loss_multiplier', 'ltv', 'bbvalue', 'apr'], dtype='str')
AN: 18 vintages


FRN: 19 vintages


STG: 19 vintages


FLD: 19 vintages


Error: 2026-05-03/2026-05-09 ENT: columns overlap but no suffix specified: Index(['loss_multiplier', 'ltv', 'bbvalue', 'apr'], dtype='str')
ENT: 18 vintages

All LOBs processed.
[PROGRESS] Scoring Complete


In [9]:
# =============================================================================
# CELL 8: DIAGNOSTICS - STEP-LEVEL ATTRIBUTION PER LOB
# =============================================================================

def build_output_df(results, wtd_mults, record_counts, ragu_gli_dict=None):
    if not results:
        return None
    step_names = list(next(iter(results.values())).keys())
    col_data = {}
    for (lob, vintage), step_dict in results.items():
        col_name = f"{lob} | {vintage}"
        col_data[col_name] = [step_dict.get(s) for s in step_names]
    diag_df = pd.DataFrame(col_data, index=step_names)
    summary = {}
    for (lob, vintage) in results.keys():
        col = f"{lob} | {vintage}"
        final_mult_mean = diag_df[col].iloc[-1]
        wtd_mult = wtd_mults.get((lob, vintage), float('nan'))
        if ragu_gli_dict is not None:
            gross_loss_impact = ragu_gli_dict.get((lob, vintage), 25 * (1 - wtd_mult))
        else:
            gross_loss_impact = 25 * (1 - wtd_mult)
        summary[col] = {
            '--- FINAL_MULT (mean)': final_mult_mean,
            '--- WTD_MULT_RAGU': wtd_mult,
            '--- GROSS_LOSS_IMPACT': gross_loss_impact,
            '--- N_RECORDS': record_counts.get((lob, vintage), 0),
        }
    summary_df = pd.DataFrame(summary)
    return pd.concat([diag_df, summary_df])


def build_flags_df(flag_results, record_counts):
    if not flag_results:
        return None
    flag_names = list(next(iter(flag_results.values())).keys())
    flag_col_data = {}
    for (lob, vintage), flag_dict in flag_results.items():
        col_name = f"{lob} | {vintage}"
        flag_col_data[col_name] = [flag_dict.get(f) for f in flag_names]
    flags_df = pd.DataFrame(flag_col_data, index=flag_names)
    n_row = {}
    for (lob, vintage) in flag_results.keys():
        n_row[f"{lob} | {vintage}"] = record_counts.get((lob, vintage), 0)
    flags_df.loc['--- N_RECORDS'] = n_row
    return flags_df


STEP_LABEL_MAP = {
    '00_initial':              'Initial (1.0)',
    '01_prev_aca_chargeoff':   'Previous ACA Chargeoff',
    '02_small_amt_financed':   'Small Amount Financed',
    '03_zero_cash_down':       'Zero Cash Down',
    '04_high_mileage':         'High Mileage Vehicle',
    '05_high_pti':             'High PTI',
    '06_car_make':             'Car Make',
    '07_theft_risk':           'Theft Risk',
    '08_mcy_low_mileage':      'MCY Low Mileage',
    '09_weekend_weekday':      'Weekend / Weekday',
    '10_student_loans':        'Student Loans',
    '11_low_pti':              'Low PTI',
    '12_chime':                'Chime / Secured Credit',
    '13_employment_type':      'Employment Type',
    '14_auth_tradelines':      'Authorized Tradelines',
    '15_fraud':                'Fraud Adjustment',
    '16_driver_flag':          'Driver Flag',
    '17_pricing_scalar':       'Dealer Level (Pricing Scalar)',
    '18_final':                'Final Clip',
}


def build_attribution_df(results, ragu_gli_dict):
    """
    Decompose gross_loss_impact across multiplier steps using logarithmic attribution.
    gross_loss_impact is sourced from get_ragu_score output to ensure the attribution
    decomposes the same value that appears in the RAGU score decomposition.
    """
    if not results:
        return None

    step_keys = list(next(iter(results.values())).keys())
    adjustment_keys = [k for k in step_keys if k != '00_initial']
    readable_labels = [STEP_LABEL_MAP.get(k, k) for k in adjustment_keys]

    col_data = {}
    for (lob, vintage), step_dict in results.items():
        col_name = f"{lob} | {vintage}"
        gross_loss_impact = ragu_gli_dict.get((lob, vintage), float('nan'))

        cumulative = [step_dict.get(k, float('nan')) for k in step_keys]
        ratios = []
        for i, k in enumerate(step_keys):
            if k == '00_initial':
                continue
            prev = cumulative[i - 1]
            curr = cumulative[i]
            if prev and prev != 0:
                ratios.append(curr / prev)
            else:
                ratios.append(1.0)

        final_mult = cumulative[-1]
        log_final = math.log(final_mult) if final_mult and final_mult > 0 and abs(final_mult - 1.0) > 1e-12 else None

        if log_final is None or pd.isna(gross_loss_impact):
            col_data[col_name] = [0.0] * len(adjustment_keys) + [gross_loss_impact if not pd.isna(gross_loss_impact) else 0.0]
        else:
            log_ratios = [math.log(r) if r and r > 0 else 0.0 for r in ratios]
            attributed = [(lr / log_final) * gross_loss_impact for lr in log_ratios]
            col_data[col_name] = attributed + [sum(attributed)]

    index_labels = readable_labels + ['--- TOTAL (check)']
    return pd.DataFrame(col_data, index=index_labels)


# --- Run diagnostics per LOB ---
diag_results_by_lob = {}

for lob in LOBS:
    ula_lob = ula_df_total[ula_df_total.lob == lob].copy()
    if len(ula_lob) == 0:
        print(f'\n{lob}: No data, skipping.')
        diag_results_by_lob[lob] = (None, None, None)
        continue

    ragu_gli_dict = {}
    model_df = results_by_lob.get(lob)
    if model_df is not None and len(model_df) > 0:
        for _, row in model_df.reset_index().iterrows():
            ragu_gli_dict[(row['lob'], row['vintage'])] = row['gross_loss_impact']

    target_vintages = sorted(ula_lob.vintage.unique())[-DIAG_N_VINTAGES:]

    lob_results = {}
    lob_flag_results = {}
    lob_wtd_mults = {}
    lob_record_counts = {}

    print(f'\n{"="*60}')
    print(f'  DIAGNOSTICS: {lob}')
    print(f'{"="*60}')

    for vintage in target_vintages:
        ula_vintage = ula_lob[ula_lob.vintage == vintage].copy()
        n = len(ula_vintage)
        if n == 0:
            continue

        print(f'\n{"="*60}')
        print(f'  {lob} {vintage}  (n={n})')
        print(f'{"="*60}')

        ula_vintage_diag, steps, flags = get_ula_multiplier_nonkmx_diag(ula_vintage, leave_out='None', verbose=True)

        diag_mix = ula_vintage_diag[['account_number', 'bbvalue', 'amt_financed', 'loss_multiplier']].copy()
        nr = new_recovery[['account_number', 'new_recovery_multiplier']].drop_duplicates(
            subset='account_number', keep='first')
        diag_mix = diag_mix.merge(nr, on='account_number', how='left').drop_duplicates(
            subset='account_number', keep='first')
        bb_pop = diag_mix[diag_mix['bbvalue'].notna() & (diag_mix['bbvalue'] > 0)]
        if len(bb_pop) > 0 and bb_pop.amt_financed.sum() > 0:
            wtd_mult = (bb_pop.loss_multiplier * bb_pop.amt_financed).sum() / bb_pop.amt_financed.sum()
        else:
            wtd_mult = float('nan')

        ragu_gli = ragu_gli_dict.get((lob, vintage), float('nan'))
        print(f"  bb_populated: {len(bb_pop)} / {n}  wtd_mult: {wtd_mult:.6f}  ragu_gli: {ragu_gli:.4f}")

        lob_results[(lob, vintage)] = steps.to_dict()
        lob_flag_results[(lob, vintage)] = flags.to_dict()
        lob_wtd_mults[(lob, vintage)] = wtd_mult
        lob_record_counts[(lob, vintage)] = n

    output_df = build_output_df(lob_results, lob_wtd_mults, lob_record_counts, ragu_gli_dict)
    flags_df = build_flags_df(lob_flag_results, lob_record_counts)
    attribution_df = build_attribution_df(lob_results, ragu_gli_dict)
    diag_results_by_lob[lob] = (output_df, flags_df, attribution_df)

    if output_df is not None:
        print(f'\n--- {lob} Multiplier Steps ---')
        display(output_df)
        print(f'\n--- {lob} Flag Means ---')
        display(flags_df)
    if attribution_df is not None:
        print(f'\n--- {lob} Gross Loss Attribution ---')
        display(attribution_df)


  DIAGNOSTICS: AN

  AN 2026-03-29/2026-04-04  (n=194)
00_initial  | loss_multiplier mean: 1.000000
01_prev_co  | flag mean: 0.010309  | loss_multiplier mean: 1.001031
02_small_af | flag mean: 0.005155  | loss_multiplier mean: 1.001031
03_zero_cd  | flag mean: 0.067010  | loss_multiplier mean: 1.001031
04_high_mi  | flag mean: 0.000000  | loss_multiplier mean: 1.001031
05_high_pti | flag mean: 0.005155  | loss_multiplier mean: 1.001031
06_car_make | penalty: 0.000000  benefit: 0.298969  | loss_multiplier mean: 0.941237
07_theft    | flag mean: 0.000000  | loss_multiplier mean: 0.941237
08_mcy_low  | flag mean: 0.000000  | loss_multiplier mean: 0.941237
09_wknd/day | weekend: 0.283505  weekday: 0.716495  | loss_multiplier mean: 0.941299
10_stud_ln  | flag mean: 0.113402  | loss_multiplier mean: 0.930082
11_low_pti  | flag mean: 0.025773  | loss_multiplier mean: 0.926307
12_chime    | flag mean: 0.195876  | loss_multiplier mean: 0.943150
13_employ   | seasonal: 0.020619  waiter: 0.01030


  AN 2026-04-19/2026-04-25  (n=198)
00_initial  | loss_multiplier mean: 1.000000
01_prev_co  | flag mean: 0.000000  | loss_multiplier mean: 1.000000
02_small_af | flag mean: 0.000000  | loss_multiplier mean: 1.000000
03_zero_cd  | flag mean: 0.126263  | loss_multiplier mean: 1.000000
04_high_mi  | flag mean: 0.000000  | loss_multiplier mean: 1.000000
05_high_pti | flag mean: 0.010101  | loss_multiplier mean: 1.000000
06_car_make | penalty: 0.000000  benefit: 0.333333  | loss_multiplier mean: 0.933333
07_theft    | flag mean: 0.000000  | loss_multiplier mean: 0.933333
08_mcy_low  | flag mean: 0.000000  | loss_multiplier mean: 0.933333
09_wknd/day | weekend: 0.287879  weekday: 0.712121  | loss_multiplier mean: 0.932838
10_stud_ln  | flag mean: 0.141414  | loss_multiplier mean: 0.923304
11_low_pti  | flag mean: 0.060606  | loss_multiplier mean: 0.914674
12_chime    | flag mean: 0.242424  | loss_multiplier mean: 0.942819
13_employ   | seasonal: 0.015152  waiter: 0.040404  | loss_multiplie

17_pricing  | scalar mean: 0.880000  | loss_multiplier mean: 0.736825
18_final    | loss_multiplier mean: 0.746718  (post all clips)
  bb_populated: 5 / 5  wtd_mult: 0.726979  ragu_gli: nan

--- AN Multiplier Steps ---


,AN | 2026-03-29/2026-04-04,AN | 2026-04-05/2026-04-11,AN | 2026-04-12/2026-04-18,AN | 2026-04-19/2026-04-25,AN | 2026-04-26/2026-05-02,AN | 2026-05-03/2026-05-09
00_initial,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
01_prev_aca_chargeoff,1.001031,1.001531,1.000521,1.000000,1.000000,1.000000
02_small_amt_financed,1.001031,1.001531,1.000521,1.000000,1.000000,1.000000
03_zero_cash_down,1.001031,1.001531,1.000521,1.000000,1.000000,1.000000
04_high_mileage,1.001031,1.002041,1.001042,1.000000,1.000893,1.000000
05_high_pti,1.001031,1.002041,1.001042,1.000000,1.000893,1.000000
06_car_make,0.941237,0.946939,0.936458,0.933333,0.940179,0.840000
07_theft_risk,0.941237,0.946939,0.936458,0.933333,0.940179,0.840000
08_mcy_low_mileage,0.941237,0.946939,0.936458,0.933333,0.940179,0.840000
09_weekend_weekday,0.941299,0.944413,0.934990,0.932838,0.946295,0.845600



--- AN Flag Means ---


...

,ENT | 2026-03-29/2026-04-04,ENT | 2026-04-05/2026-04-11,ENT | 2026-04-12/2026-04-18,ENT | 2026-04-19/2026-04-25,ENT | 2026-04-26/2026-05-02,ENT | 2026-05-03/2026-05-09
00_initial,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
01_prev_aca_chargeoff,1.000699,1.001887,1.002475,1.003429,1.000758,1.000000
02_small_amt_financed,1.000699,1.001887,1.002475,1.003429,1.000758,1.000000
03_zero_cash_down,1.000699,1.001887,1.002475,1.003429,1.000758,1.000000
04_high_mileage,1.000699,1.001887,1.002475,1.003429,1.000758,1.000000
05_high_pti,1.000699,1.001887,1.002475,1.003429,1.000758,1.000000
06_car_make,0.949650,0.951792,0.944950,0.954286,0.944697,0.900000
07_theft_risk,0.949650,0.951792,0.944950,0.954286,0.944697,0.900000
08_mcy_low_mileage,0.949650,0.951792,0.944950,0.954286,0.944697,0.900000
09_weekend_weekday,0.956357,0.961385,0.949156,0.960691,0.959455,0.918000



--- ENT Flag Means ---


,ENT | 2026-03-29/2026-04-04,ENT | 2026-04-05/2026-04-11,ENT | 2026-04-12/2026-04-18,ENT | 2026-04-19/2026-04-25,ENT | 2026-04-26/2026-05-02,ENT | 2026-05-03/2026-05-09
prev_co_flag,0.006993,0.018868,0.024752,0.034286,0.007576,0.000
small_amt_financed_flag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000
pricing_change_flag,1.000000,1.000000,1.000000,1.000000,1.000000,1.000
zero_cash_down_flag,0.115385,0.075472,0.079208,0.142857,0.136364,0.500
high_mileage_vehicle_flag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000
high_pti_flag,0.010490,0.018868,0.004950,0.005714,0.007576,0.000
car_make_penalty_flag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000
car_make_benefit_flag,0.255245,0.250000,0.287129,0.245714,0.280303,0.500
theft_risk_flag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000
mcy_low_mileage_flag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000



--- ENT Gross Loss Attribution ---


,ENT | 2026-03-29/2026-04-04,ENT | 2026-04-05/2026-04-11,ENT | 2026-04-12/2026-04-18,ENT | 2026-04-19/2026-04-25,ENT | 2026-04-26/2026-05-02,ENT | 2026-05-03/2026-05-09
Previous ACA Chargeoff,-0.027023,-0.085311,-0.069391,-0.213368,-0.014020,0.0
Small Amount Financed,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0
Zero Cash Down,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0
High Mileage Vehicle,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0
High PTI,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0
Car Make,2.024051,2.321396,1.658722,3.130339,1.067275,0.0
Theft Risk,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0
MCY Low Mileage,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.0
Weekend / Weekday,-0.272024,-0.453834,-0.124656,-0.417056,-0.286973,0.0
Student Loans,0.259078,0.315716,0.166832,0.847219,0.154498,0.0


In [10]:
# =============================================================================
# CELL 9: EXCEL EXPORT
# =============================================================================

with pd.ExcelWriter(EXCEL_OUTPUT, engine='openpyxl') as writer:
    # Per-LOB diagnostic sheets
    for lob in LOBS:
        diag_data = diag_results_by_lob.get(lob, (None, None, None))
        output_df, flags_df, attribution_df = diag_data

        sheet_mult = f'{lob} Mult Steps'[:31]
        sheet_flags = f'{lob} Flags'[:31]
        sheet_attr = f'{lob} Attribution'[:31]

        if output_df is not None:
            output_df.to_excel(writer, sheet_name=sheet_mult)
        if flags_df is not None:
            flags_df.to_excel(writer, sheet_name=sheet_flags)
        if attribution_df is not None:
            attribution_df.to_excel(writer, sheet_name=sheet_attr)

    # Summary sheet with latest vintage per LOB from RAGU output
    summary_rows = []
    for lob in LOBS:
        model_df = results_by_lob.get(lob)
        if model_df is not None and len(model_df) > 0:
            lob_rows = model_df.reset_index()
            lob_rows = lob_rows[lob_rows.lob == lob]
            if len(lob_rows) > 0:
                last_vintage = lob_rows.vintage.max()
                last_row = lob_rows[lob_rows.vintage == last_vintage].iloc[0]
                summary_rows.append({
                    'LOB': lob,
                    'Latest Vintage': last_vintage,
                    'Contract MS': last_row.get('ms_original', float('nan')),
                    'Gross Loss Impact': last_row.get('gross_loss_impact', float('nan')),
                    'Recovery Impact': last_row.get('recovery_impact', float('nan')),
                    'LTV Impact': last_row.get('ltv_impact', float('nan')),
                    'APR Impact': last_row.get('apr_impact', float('nan')),
                    'RAGU Score': last_row.get('ragu_score', float('nan')),
                    'LTV': last_row.get('ltv', float('nan')),
                    'Loss Multiplier': last_row.get('loss_multiplier', float('nan')),
                })
    if summary_rows:
        summary_df = pd.DataFrame(summary_rows).set_index('LOB')
        summary_df.to_excel(writer, sheet_name='Summary')
        display(summary_df)

print(f"\nExported to {EXCEL_OUTPUT}")
print("[PROGRESS] Export Complete")


,Latest Vintage,Contract MS,Gross Loss Impact,Recovery Impact,LTV Impact,APR Impact,RAGU Score,LTV,Loss Multiplier
LOB,,,,,,,,,
AN,2026-04-26/2026-05-02,141.587233,3.389160,3.719510,4.481750,1.716585,154.894238,1.535257,0.864434
FRN,2026-05-03/2026-05-09,143.124208,3.100894,-1.104042,4.743006,-0.017608,149.846458,1.516810,0.875964
STG,2026-05-03/2026-05-09,142.529574,3.871103,14.115735,5.105693,1.830841,167.452946,1.491923,0.845156
FLD,2026-05-03/2026-05-09,140.234221,4.696566,-1.224243,1.121377,0.071818,144.899739,1.360272,0.812137
ENT,2026-04-26/2026-05-02,140.881045,-0.221324,2.353324,2.373362,-0.069050,145.317356,1.272366,1.008853



Exported to nonkmx_gl_diagnostics.xlsx
[PROGRESS] Export Complete
